# 2. Self-Attention

Self-attention is when a sequence attends to itself. Each position can attend to all positions, including itself!
This is the building block of transformers.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


## Self-Attention: Q, K, V from Same Input

In self-attention, Q, K, and V all come from the same input sequence.
We use learned linear transformations to create Q, K, V from the input.


In [ ]:
# Step 1: Create Q, K, V from input
# defining sequence_length and dimension_model
seq_len, d_model = 5, 8

# Input sequence (e.g., word embeddings)
# Creating word embeddings input
x = torch.randn(seq_len, d_model)
print(f"Input shape: {x.shape}")

# Linear transformations to create Q, K, V
# Use linear transformations matrix for Q, K and V
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

# Transform the word embedding input into linear fx for Q,K and V
Q = W_q(x)  # Query
K = W_k(x)  # Key
V = W_v(x)  # Value

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print("\nIn self-attention, Q, K, V all come from the same input x!")


Input shape: torch.Size([5, 8])
Q shape: torch.Size([5, 8])
K shape: torch.Size([5, 8])
V shape: torch.Size([5, 8])

In self-attention, Q, K, V all come from the same input x!


## Self-Attention Layer

Let's build a complete self-attention layer from scratch!


In [ ]:
class SelfAttention(nn.Module):
    """Self-attention layer from scratch"""
    
    # inheriting properties of pytorch and NN
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        
        # Linear layers to create Q, K, V
        # to transform word embedding inputs into before processing
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x):
        """
        Args:
            x: Input tensor [batch_size, seq_len, d_model]
        Returns:
            output: [batch_size, seq_len, d_model]
            attention_weights: [batch_size, seq_len, seq_len]
        """
        # define dimension of x(input) same for
        #   batch_size, sequence_length, dimension_model
        batch_size, seq_len, d_model = x.shape
        
        # Create Q, K, V
        # Transform the word embedding inputs into linear transormation
        Q = self.W_q(x)  # [batch_size, seq_len, d_model]
        K = self.W_k(x)  # [batch_size, seq_len, d_model]
        V = self.W_v(x)  # [batch_size, seq_len, d_model]
        
        # Compute attention score: First step
        # Compute attention scores: QK^T
        scores = torch.matmul(Q, K.transpose(-2, -1))  # [batch_size, seq_len, seq_len]
        
        # Compute attention score: Second step
        # Scale by sqrt(d_model) 
        scores = scores / np.sqrt(d_model)

        # Compute attention weight: Third step
        # Softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)  # [batch_size, seq_len, seq_len]

        # Calculate output: Fourth step 
        # Apply attention to values
        output = torch.matmul(attention_weights, V)  # [batch_size, seq_len, d_model]
        
        return output, attention_weights

# Test the self-attention layer
batch_size = 2 
seq_len = 10 
d_model = 64

self_attn = SelfAttention(d_model)
x = torch.randn(batch_size, seq_len, d_model)

output, attn_weights = self_attn(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nAttention weights for first sequence, first position:")
# accessing element in innermost layer if the specified not as same specified
#   Example: supposedly attn_weight dimension is (x, y, z) =  (2, 10, 10)
#   But, we can extract by not fully specifying from the innermost
#       Example, (y,z) = (0,0); x will be default as first
print(attn_weights[0, 0]) 
print(f"Sum: {attn_weights[0, 0].sum():.3f} (should be 1.0)")


Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])
Attention weights shape: torch.Size([2, 10, 10])

Attention weights for first sequence, first position:
tensor([0.0962, 0.1206, 0.0619, 0.1122, 0.0982, 0.0839, 0.0979, 0.0785, 0.0662,
        0.1844], grad_fn=<SelectBackward0>)
Sum: 1.000 (should be 1.0)
